In [1]:

# ============================================================
# Stage 02: Data Preprocessing
# Step 2.1: Imports & Configuration
# ============================================================

from pathlib import Path
from zipfile import ZipFile
import pandas as pd

# 1. Automatically locate the project
CURRENT_DIR = Path.cwd().resolve()

PROJECT_DIR = next(
    (
        path for path in [CURRENT_DIR, *CURRENT_DIR.parents]
        if (path / "data" / "raw" / "FLAT_RCL_POST_2010.zip").exists()
    ),
    None
)

if PROJECT_DIR is None:
    raise FileNotFoundError(
        f"Dataset not found. Current directory: {CURRENT_DIR}"
    )

# 2. Define paths
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

ZIP_PATH = RAW_DIR / "FLAT_RCL_POST_2010.zip"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# 3. Define our five target classes
TARGET_CLASSES = {
    "ELECTRICAL SYSTEM": "Electrical System",
    "AIR BAGS": "Air Bags",
    "SERVICE BRAKES": "Service Brakes",
    "STEERING": "Steering",
    "FUEL SYSTEM": "Fuel System"
}

print("=" * 50)
print("STAGE 02 - DATA PREPROCESSING")
print("=" * 50)

print("Project Directory:", PROJECT_DIR)
print("Dataset Found:", ZIP_PATH.exists())
print("Target Classes:", len(TARGET_CLASSES))

for label in TARGET_CLASSES.values():
    print("-", label)


STAGE 02 - DATA PREPROCESSING
Project Directory: C:\Users\faisa\Desktop\NLP Project
Dataset Found: True
Target Classes: 5
- Electrical System
- Air Bags
- Service Brakes
- Steering
- Fuel System


In [2]:
# ============================================================
# Step 2.1: Load & Map Target Classes
# ============================================================

target_chunks = []

# Function to map detailed components to target labels
def map_component(component_name):

    for prefix, label in TARGET_CLASSES.items():
        if component_name.startswith(prefix):
            return label

    return None


# Read NHTSA dataset from ZIP
with ZipFile(ZIP_PATH, "r") as archive:

    txt_name = next(
        name for name in archive.namelist()
        if name.lower().endswith(".txt")
    )

    with archive.open(txt_name) as file:

        reader = pd.read_csv(
            file,
            sep="\t",
            header=None,
            usecols=[1, 6, 10, 19],
            encoding="latin-1",
            dtype=str,
            keep_default_na=False,
            chunksize=25000
        )

        for chunk in reader:

            # Assign column names
            chunk.columns = [
                "CAMPNO",
                "COMPNAME",
                "RCLTYPECD",
                "DESC_DEFECT"
            ]

            # Remove surrounding whitespace
            for col in chunk.columns:
                chunk[col] = chunk[col].str.strip()

            # Standardize component names
            chunk["COMPNAME"] = (
                chunk["COMPNAME"].str.upper()
            )

            # Keep vehicle recalls only
            chunk = chunk[
                chunk["RCLTYPECD"].str.upper() == "V"
            ].copy()

            # Map components into five classes
            chunk["label"] = (
                chunk["COMPNAME"].apply(map_component)
            )

            # Keep only selected target classes
            chunk = chunk.dropna(subset=["label"])

            target_chunks.append(chunk)


# Combine all chunks
df_target = pd.concat(
    target_chunks,
    ignore_index=True
)

# ============================================================
# Display Results
# ============================================================

print("=" * 50)
print("LABEL MAPPING RESULTS")
print("=" * 50)

print("Total Target Records:", len(df_target))

print(
    "Unique Campaigns:",
    df_target["CAMPNO"].nunique()
)

print(
    "Number of Classes:",
    df_target["label"].nunique()
)

print("\nRecords by Target Class:")
print(df_target["label"].value_counts())

print("\nFirst 5 Records:")

display(
    df_target[
        ["CAMPNO", "COMPNAME", "label", "DESC_DEFECT"]
    ].head()
)


LABEL MAPPING RESULTS
Total Target Records: 89857
Unique Campaigns: 5261
Number of Classes: 5

Records by Target Class:
label
Electrical System    44208
Fuel System          14666
Air Bags             13158
Service Brakes       10779
Steering              7046
Name: count, dtype: int64

First 5 Records:


,CAMPNO,COMPNAME,label,DESC_DEFECT
0,10V589000,ELECTRICAL SYSTEM:WIRING:FUSES AND CIRCUIT BRE...,Electrical System,DUTCHMEN IS RECALLING CERTAIN MODEL YEAR 2011 ...
1,10V589000,ELECTRICAL SYSTEM:WIRING:FUSES AND CIRCUIT BRE...,Electrical System,DUTCHMEN IS RECALLING CERTAIN MODEL YEAR 2011 ...
2,10V589000,ELECTRICAL SYSTEM:WIRING:FUSES AND CIRCUIT BRE...,Electrical System,DUTCHMEN IS RECALLING CERTAIN MODEL YEAR 2011 ...
3,10V589000,ELECTRICAL SYSTEM:WIRING:FUSES AND CIRCUIT BRE...,Electrical System,DUTCHMEN IS RECALLING CERTAIN MODEL YEAR 2011 ...
4,10V589000,ELECTRICAL SYSTEM:WIRING:FUSES AND CIRCUIT BRE...,Electrical System,DUTCHMEN IS RECALLING CERTAIN MODEL YEAR 2011 ...


In [3]:
# ============================================================
# Step 2.2 - Remove Conflicting Campaigns
# ============================================================

# 1. Count unique labels for each campaign
campaign_label_counts = (
    df_target.groupby("CAMPNO")["label"].nunique()
)

# 2. Find valid campaigns (only one target label)
valid_campaign_ids = campaign_label_counts[
    campaign_label_counts == 1
].index

# 3. Find conflicting campaigns (multiple target labels)
conflicting_campaign_ids = campaign_label_counts[
    campaign_label_counts > 1
].index

# 4. Remove all records belonging to conflicting campaigns
df_valid = df_target[
    df_target["CAMPNO"].isin(valid_campaign_ids)
].copy()

# 5. Verify results
print("=" * 55)
print("CONFLICT REMOVAL RESULTS")
print("=" * 55)

print("Original Records:", len(df_target))
print("Original Campaigns:", df_target["CAMPNO"].nunique())

print("\nConflicting Campaigns:", len(conflicting_campaign_ids))

print("\nRemaining Records:", len(df_valid))
print("Remaining Campaigns:", df_valid["CAMPNO"].nunique())

# 6. Count unique campaigns for each class
campaign_distribution = (
    df_valid.drop_duplicates(subset=["CAMPNO"])
    ["label"]
    .value_counts()
)

print("\nValid Campaigns by Class:")
print(campaign_distribution)

# 7. Check for remaining conflicts
remaining_conflicts = (
    df_valid.groupby("CAMPNO")["label"].nunique() > 1
).sum()

print("\nRemaining Conflicts:", remaining_conflicts)

assert remaining_conflicts == 0, "Conflicts still exist!"
assert df_valid["CAMPNO"].nunique() == len(valid_campaign_ids)

print("\nConflict removal completed successfully!")


CONFLICT REMOVAL RESULTS
Original Records: 89857
Original Campaigns: 5261

Conflicting Campaigns: 165

Remaining Records: 84837
Remaining Campaigns: 5096

Valid Campaigns by Class:
label
Electrical System    1896
Service Brakes        982
Air Bags              789
Fuel System           738
Steering              691
Name: count, dtype: int64

Remaining Conflicts: 0

Conflict removal completed successfully!


In [4]:
# ============================================================
# Step 2.3A - Duplicate & Text Quality Analysis
# ============================================================

# Create a copy for inspection
df_analysis = df_valid[
    ["CAMPNO", "DESC_DEFECT", "label"]
].copy()

# Normalize whitespace in text
df_analysis["text_clean"] = (
    df_analysis["DESC_DEFECT"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Normalized key for detecting duplicates
df_analysis["text_key"] = (
    df_analysis["text_clean"].str.casefold()
)

print("=" * 55)
print("DUPLICATE & TEXT QUALITY ANALYSIS")
print("=" * 55)

# 1. Basic statistics
print("Total Records:", len(df_analysis))
print("Unique Campaigns:", df_analysis["CAMPNO"].nunique())

# 2. Check empty descriptions
empty_texts = (
    df_analysis["text_clean"].str.len() == 0
).sum()

print("\nEmpty Descriptions:", empty_texts)

# 3. Count exact duplicate rows
duplicate_rows = df_analysis.duplicated(
    subset=["CAMPNO", "text_key", "label"]
).sum()

print("Duplicate Campaign-Text Rows:", duplicate_rows)

# 4. Check campaigns with different descriptions
campaign_text_counts = (
    df_analysis.groupby("CAMPNO")["text_key"].nunique()
)

multiple_text_campaigns = (
    campaign_text_counts > 1
).sum()

print(
    "Campaigns With Multiple Different Descriptions:",
    multiple_text_campaigns
)

# 5. Check texts shared by different campaigns
text_campaign_counts = (
    df_analysis.groupby("text_key")["CAMPNO"].nunique()
)

shared_texts = text_campaign_counts[
    text_campaign_counts > 1
]

print(
    "Descriptions Shared Across Campaigns:",
    len(shared_texts)
)

# 6. Check identical texts with different labels
text_label_counts = (
    df_analysis.groupby("text_key")["label"].nunique()
)

ambiguous_texts = text_label_counts[
    text_label_counts > 1
]

print(
    "Identical Descriptions With Different Labels:",
    len(ambiguous_texts)
)

# 7. Text length statistics
text_lengths = df_analysis["text_clean"].str.len()

print("\nText Length Statistics:")
print("Minimum:", text_lengths.min())
print("Median:", text_lengths.median())
print("Maximum:", text_lengths.max())

# 8. Examples of descriptions shared across campaigns
print("\nExamples of Shared Descriptions:")

for text_key, count in (
    shared_texts.sort_values(ascending=False).head(5).items()
):
    labels = sorted(
        df_analysis.loc[
            df_analysis["text_key"] == text_key,
            "label"
        ].unique()
    )

    print("-" * 50)
    print("Campaigns:", count)
    print("Labels:", labels)
    print("Text:", text_key[:150])


DUPLICATE & TEXT QUALITY ANALYSIS
Total Records: 84837
Unique Campaigns: 5096

Empty Descriptions: 0
Duplicate Campaign-Text Rows: 79741
Campaigns With Multiple Different Descriptions: 0
Descriptions Shared Across Campaigns: 13
Identical Descriptions With Different Labels: 0

Text Length Statistics:
Minimum: 110
Median: 410.0
Maximum: 1972

Examples of Shared Descriptions:
--------------------------------------------------
Campaigns: 2
Labels: ['Fuel System']
Text: daimler trucks north america (daimler) is recalling certain model year 2013-2014 saf-t-liner (ef, efx, and hdx) buses equipped with an auxiliary fuel 
--------------------------------------------------
Campaigns: 2
Labels: ['Electrical System']
Text: ford motor company (ford) is recalling certain 2015-2017 ford transit vehicles, equipped with a trailer tow module. water can enter the tow modules an
--------------------------------------------------
Campaigns: 2
Labels: ['Electrical System']
Text: ford motor company (ford) is

In [5]:
# ============================================================
# Step 2.3B - Deduplication & Text Cleaning
# ============================================================

# 1. Verify each campaign has only one unique description
campaign_text_counts = (
    df_analysis.groupby("CAMPNO")["text_key"].nunique()
)

assert (campaign_text_counts == 1).all(), (
    "Some campaigns have multiple descriptions!"
)

# 2. Sort campaigns for reproducible selection
df_sorted = df_analysis.sort_values(
    by=["CAMPNO", "text_key"],
    kind="mergesort"
)

# 3. Keep exactly one record per campaign
df_campaign = df_sorted.drop_duplicates(
    subset=["CAMPNO"],
    keep="first"
).copy()

print("=" * 55)
print("CAMPAIGN DEDUPLICATION")
print("=" * 55)

print("Before:", len(df_analysis))
print("After:", len(df_campaign))
print(
    "Removed:",
    len(df_analysis) - len(df_campaign)
)

# 4. Detect identical texts with conflicting labels
label_counts = (
    df_campaign.groupby("text_key")["label"].nunique()
)

assert (label_counts == 1).all(), (
    "Identical descriptions have conflicting labels!"
)

# 5. Remove descriptions shared across campaigns
df_clean = df_campaign.drop_duplicates(
    subset=["text_key"],
    keep="first"
).copy()

print("\n" + "=" * 55)
print("CROSS-CAMPAIGN DEDUPLICATION")
print("=" * 55)

print("Before:", len(df_campaign))
print("After:", len(df_clean))
print(
    "Removed:",
    len(df_campaign) - len(df_clean)
)

# 6. Keep final required columns
df_clean = df_clean[
    ["CAMPNO", "text_clean", "label"]
].rename(
    columns={"text_clean": "text"}
).reset_index(drop=True)

# 7. Final verification
assert df_clean["CAMPNO"].is_unique
assert df_clean["text"].str.len().gt(0).all()
assert df_clean["label"].nunique() == 5

assert not (
    df_clean["text"]
    .str.casefold()
    .duplicated()
    .any()
)

print("\n" + "=" * 55)
print("CLEAN DATASET SUMMARY")
print("=" * 55)

print("Final Examples:", len(df_clean))
print("Unique Campaigns:", df_clean["CAMPNO"].nunique())
print("Unique Classes:", df_clean["label"].nunique())

print("\nExamples by Class:")
print(df_clean["label"].value_counts())

print("\nSample Records:")
display(df_clean.head())

print("\nAll verification checks passed!")


CAMPAIGN DEDUPLICATION
Before: 84837
After: 5096
Removed: 79741

CROSS-CAMPAIGN DEDUPLICATION
Before: 5096
After: 5083
Removed: 13

CLEAN DATASET SUMMARY
Final Examples: 5083
Unique Campaigns: 5083
Unique Classes: 5

Examples by Class:
label
Electrical System    1890
Service Brakes        979
Air Bags              788
Fuel System           737
Steering              689
Name: count, dtype: int64

Sample Records:


,CAMPNO,text,label
0,10V001000,HONDA IS RECALLING CERTAIN MODEL YEAR 2006 AND...,Electrical System
1,10V002000,HONDA IS RECALLING CERTAIN MODEL YEAR ACURA 20...,Air Bags
2,10V005000,NISSAN DIESEL IS RECALLING CERTAIN MODEL YEAR ...,Service Brakes
3,10V006000,NISSAN DIESEL IS RECALLING CERTAIN MODEL YEAR ...,Service Brakes
4,10V007000,HYUNDAI DE PUERTO RICO IS RECALLING CERTAIN MO...,Air Bags



All verification checks passed!


In [6]:
# ============================================================
# Step 2.4 - Train / Development / Test Split
# ============================================================

from sklearn.model_selection import train_test_split
import pandas as pd

RANDOM_STATE = 42

# 1. Split into 70% Training and 30% Temporary
train_df, temp_df = train_test_split(
    df_clean,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=df_clean["label"]
)

# 2. Split temporary data into 15% Dev and 15% Test
dev_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temp_df["label"]
)

# 3. Reset indices
train_df = train_df.reset_index(drop=True)
dev_df = dev_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# ============================================================
# Verify dataset sizes
# ============================================================

print("=" * 55)
print("DATASET SPLITTING RESULTS")
print("=" * 55)

print("Total Examples:", len(df_clean))
print("Training Examples:", len(train_df))
print("Development Examples:", len(dev_df))
print("Test Examples:", len(test_df))

print("\nSplit Percentages:")

for name, subset in [
    ("Training", train_df),
    ("Development", dev_df),
    ("Test", test_df)
]:
    percentage = len(subset) / len(df_clean) * 100
    print(f"{name}: {percentage:.2f}%")

# ============================================================
# Class Distribution
# ============================================================

distribution = pd.DataFrame({
    "Training": train_df["label"].value_counts(),
    "Development": dev_df["label"].value_counts(),
    "Test": test_df["label"].value_counts()
}).fillna(0).astype(int)

print("\nClass Distribution:")
display(distribution)

# ============================================================
# Check Data Leakage
# ============================================================

def check_overlap(df1, df2, column):
    return len(
        set(df1[column]) & set(df2[column])
    )

print("\n" + "=" * 55)
print("DATA LEAKAGE VERIFICATION")
print("=" * 55)

# Campaign overlap
train_dev_campaign = check_overlap(
    train_df, dev_df, "CAMPNO"
)

train_test_campaign = check_overlap(
    train_df, test_df, "CAMPNO"
)

dev_test_campaign = check_overlap(
    dev_df, test_df, "CAMPNO"
)

print("\nCampaign Overlap:")
print("Train / Dev:", train_dev_campaign)
print("Train / Test:", train_test_campaign)
print("Dev / Test:", dev_test_campaign)

# Exact normalized text overlap
def text_keys(df):
    return set(
        df["text"].str.strip().str.casefold()
    )

train_keys = text_keys(train_df)
dev_keys = text_keys(dev_df)
test_keys = text_keys(test_df)

print("\nText Overlap:")
print("Train / Dev:", len(train_keys & dev_keys))
print("Train / Test:", len(train_keys & test_keys))
print("Dev / Test:", len(dev_keys & test_keys))

# ============================================================
# Final Assertions
# ============================================================

assert (
    len(train_df) + len(dev_df) + len(test_df)
    == len(df_clean)
)

assert train_dev_campaign == 0
assert train_test_campaign == 0
assert dev_test_campaign == 0

assert len(train_keys & dev_keys) == 0
assert len(train_keys & test_keys) == 0
assert len(dev_keys & test_keys) == 0

assert train_df["label"].nunique() == 5
assert dev_df["label"].nunique() == 5
assert test_df["label"].nunique() == 5

print("\nAll splitting verification checks passed!")


DATASET SPLITTING RESULTS
Total Examples: 5083
Training Examples: 3558
Development Examples: 762
Test Examples: 763

Split Percentages:
Training: 70.00%
Development: 14.99%
Test: 15.01%

Class Distribution:


,Training,Development,Test
label,,,
Electrical System,1323,283,284
Service Brakes,685,147,147
Air Bags,552,118,118
Fuel System,516,110,111
Steering,482,104,103



DATA LEAKAGE VERIFICATION

Campaign Overlap:
Train / Dev: 0
Train / Test: 0
Dev / Test: 0

Text Overlap:
Train / Dev: 0
Train / Test: 0
Dev / Test: 0

All splitting verification checks passed!


In [9]:

# ============================================================
# Step 2.5 - Save Processed Datasets
# ============================================================

from pathlib import Path
import pandas as pd

# 1. Define output directory
OUTPUT_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 2. Define datasets
datasets = {
    "train": train_df,
    "dev": dev_df,
    "test": test_df
}

# 3. Save CSV files
for name, dataframe in datasets.items():

    output_path = OUTPUT_DIR / f"{name}.csv"

    dataframe.to_csv(
        output_path,
        index=False,
        encoding="utf-8"
    )

    print(f"Saved: {output_path.name}")
    print(f"Rows: {len(dataframe)}")
    print("-" * 40)

print("\nAll datasets saved successfully!")


Saved: train.csv
Rows: 3558
----------------------------------------
Saved: dev.csv
Rows: 762
----------------------------------------
Saved: test.csv
Rows: 763
----------------------------------------

All datasets saved successfully!


In [10]:
# ============================================================
# Step 2.5 - Verify Saved Datasets
# ============================================================

from pandas.testing import assert_frame_equal

print("=" * 55)
print("SAVED DATASET VERIFICATION")
print("=" * 55)

loaded_datasets = {}

for name, original_df in datasets.items():

    file_path = OUTPUT_DIR / f"{name}.csv"

    # Read saved dataset
    loaded_df = pd.read_csv(
        file_path,
        dtype=str,
        keep_default_na=False,
        encoding="utf-8"
    )

    loaded_datasets[name] = loaded_df

    # Verify data matches original
    assert_frame_equal(
        original_df.reset_index(drop=True),
        loaded_df.reset_index(drop=True)
    )

    print(f"{name}.csv")
    print("Rows:", len(loaded_df))
    print("Columns:", list(loaded_df.columns))
    print("File size (KB):",
          round(file_path.stat().st_size / 1024, 2))
    print("Status: PASS")
    print("-" * 40)

# ============================================================
# Verify No Data Leakage After Loading
# ============================================================

train = loaded_datasets["train"]
dev = loaded_datasets["dev"]
test = loaded_datasets["test"]

for column in ["CAMPNO", "text"]:

    train_values = set(train[column])
    dev_values = set(dev[column])
    test_values = set(test[column])

    assert not (train_values & dev_values)
    assert not (train_values & test_values)
    assert not (dev_values & test_values)

# Verify total rows
total = sum(len(df) for df in loaded_datasets.values())

assert total == len(df_clean)

print("\nTotal Examples:", total)
print("Data Leakage: NONE (exact overlap)")
print("Saved Data Integrity: PASS")

print("\nSTAGE 02 COMPLETED SUCCESSFULLY!")


SAVED DATASET VERIFICATION
train.csv
Rows: 3558
Columns: ['CAMPNO', 'text', 'label']
File size (KB): 1245.95
Status: PASS
----------------------------------------
dev.csv
Rows: 762
Columns: ['CAMPNO', 'text', 'label']
File size (KB): 266.23
Status: PASS
----------------------------------------
test.csv
Rows: 763
Columns: ['CAMPNO', 'text', 'label']
File size (KB): 266.0
Status: PASS
----------------------------------------

Total Examples: 5083
Data Leakage: NONE (exact overlap)
Saved Data Integrity: PASS

STAGE 02 COMPLETED SUCCESSFULLY!
